# RAGAS feature extraction on Merged ASQA/MS MARCO/WikiEval dataset

Notebook này tạo feature table theo từng sample từ `labeled_merged.csv`, chỉ dùng các metric black-box của RAGAS khi **không có ground truth**

Output:
- 3 lần chạy feature extractions với cùng 1 data gốc, 1 prompt và 1 model llm: gpt-4o-mini
- So sánh và phân tích sự sai khác nhau giữa 3 lần chạy, nếu có, để hiểu rõ hơn về sự ổn định của RAGAS khi không có ground truth.

## Imports and setup

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import ast
import importlib
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import seaborn as sns
import matplotlib.pyplot as plt

from dotenv import load_dotenv
import os
sys.path.append('..')

import src.filtering.ragas as _ragas_mod
importlib.reload(_ragas_mod)

from src.filtering.ragas import RAGAS
from src.filtering.ragas_feature_extractor import RagasFeatureExtractor
from src.evaluation import plot_evaluation_results

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
np.random.seed(42)

c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = Path('../data/labeled_merged.csv')
OUTPUT_DIR = Path('../results/ragas_filter/merged')
N_TIMES = 3

## Imports data and initialize evaluator

In [3]:
df = pd.read_csv(DATA_PATH)
df["label"] = df["label"].astype(int)

expected_columns = {"id", "question", "answer", "context", "label"}
missing_columns = expected_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in CSV: {sorted(missing_columns)}")

print(df.shape)
display(df.head())
print(df["label"].value_counts(dropna=False).sort_index())

(9806, 6)


,id,question,context,answer,label,dataset
0,asqa_0,When does the new bunk'd come out?,"- (List of Bunk'd episodes) The new bunk'd episode 41 comes out on April 21, 2017, episode 42 comes out on April 28,...","The new bunk'd episode 41 comes out on April 21, 2017, episode 42 comes out on April 28, 2017 and episode 42 is due ...",1,asqa
1,asqa_1,Who won the 2016 ncaa football national championship?,- (2016 College Football Playoff National Championship) The 2015 - 2016 season's ncaa national football championship...,The 2015 - 2016 season's ncaa national football championship game was played between the Clemson Tigers and the Alab...,1,asqa
2,asqa_2,When was the last time the death penalty was used in pa?,"- (QA_1) As of 2017, when was the last time the death penalty was carried out in PA? July 6, 1999.\r\n- (QA_2) As of...","The last time the death penalty was used in pa was on July 6, 1999.",1,asqa
3,asqa_3,Where will failure of the left ventricle cause increased pressure?,"- (Heart failure) ""Backward"" failure of the left ventricle causes congestion of the lungs' blood vessels, and theref...","""Backward"" failure of the left ventricle causes congestion of the lungs' blood vessels, and therefore causes increas...",1,asqa
4,asqa_4,Who won the war between ethiopia and italy?,- (Second Italo-Ethiopian War) The first war between Italy and Ethiopia took place from 1895 to 1896. This war was w...,The first war between Italy and Ethiopia took place from 1895 to 1896. This war was won by the Ethiopian army after ...,1,asqa


label
0    4903
1    4903
Name: count, dtype: int64


## RAGAs feature extraction

In [4]:
# Black-box RAGAS metrics: no ground-truth dependent scores
# We intentionally skip answer_correctness / answer_similarity.
metric_names = [
    "faithfulness",
    "answer_relevancy",
    "context_relevancy",
]

evaluator = RAGAS(
    metrics=metric_names,

    # OpenAI models
    llm_model="gpt-4o-mini",
    embedding_model="text-embedding-3-small",

    # optional
    api_key=OPENAI_API_KEY,

    # optional
    temperature=0,
)

for i in range(N_TIMES):
    print(f"\n=== RAGAS feature extraction: Run {i+1}/{N_TIMES} ===")
    feature_path = OUTPUT_DIR / f"merged_ragas_features_{i+1}.csv"
    checkpoint_path = OUTPUT_DIR / f"merged_ragas_checkpoints_{i+1}.csv"
    extractor = RagasFeatureExtractor(ragas_evaluator=evaluator, feature_cols=metric_names)
    feature_df = extractor.transform(data=df, 
                                    feature_path=feature_path,
                                    checkpoint_path=checkpoint_path)
    print(f"> Save RAGAS features to {feature_path}")
    print(feature_df.shape)
    display(feature_df.describe().T)
    display(feature_df.head())


=== RAGAS feature extraction: Run 1/3 ===
Resuming from sample 9806
> Save RAGAS features to ..\results\ragas_filter\merged\merged_ragas_features_1.csv
(9806, 6)


,count,mean,std,min,25%,50%,75%,max
faithfulness,9806.0,0.681684,0.345794,0.0,0.400000,0.800000,1.000000,1.0
answer_relevancy,9806.0,0.732503,0.211706,0.0,0.668929,0.786409,0.868929,1.0
context_relevancy,9806.0,0.204840,0.179689,0.0,0.090909,0.142857,0.250000,1.0
label,9806.0,0.500000,0.500025,0.0,0.000000,0.500000,1.000000,1.0


,id,faithfulness,answer_relevancy,context_relevancy,label,dataset
0,asqa_0,0.666667,0.726235,0.090909,1,asqa
1,asqa_1,1.000000,0.957490,0.111111,1,asqa
2,asqa_2,1.000000,0.965820,0.111111,1,asqa
3,asqa_3,1.000000,0.641608,0.125000,1,asqa
4,asqa_4,1.000000,0.800938,0.214286,1,asqa



=== RAGAS feature extraction: Run 2/3 ===
Resuming from sample 9806
> Save RAGAS features to ..\results\ragas_filter\merged\merged_ragas_features_2.csv
(9806, 6)


,count,mean,std,min,25%,50%,75%,max
faithfulness,9805.0,0.679689,0.346097,0.0,0.400000,0.800000,1.000000,1.0
answer_relevancy,9806.0,0.721616,0.214478,0.0,0.654541,0.773243,0.861250,1.0
context_relevancy,9806.0,0.331757,0.265949,0.0,0.142857,0.250000,0.428571,1.0
label,9806.0,0.500000,0.500025,0.0,0.000000,0.500000,1.000000,1.0


...

Saved checkpoint: 3350/9806


Evaluating: 100%|██████████| 150/150 [08:50<00:00,  3.54s/it] 


Saved checkpoint: 3400/9806


Evaluating:   1%|▏         | 2/150 [00:38<47:51, 19.40s/it]
Exception in thread Thread-8:
Traceback (most recent call last):
  File "c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\httpx\_transports\default.py", line 101, in map_httpcore_exceptions
    yield
  File "c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\httpx\_transports\default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\httpcore\_sync\connection_pool.py", line 256, in handle_request
    raise exc from None
  File "c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\httpcore\_sync\connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
  File "c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\httpcore\_sync\connection.py", line 101, in handle_request
    raise exc
  File "c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\httpcore\_sync\connection.py", li

ExceptionInRunner: The runner thread which was running the jobs raised an exeception. Read the traceback above to debug it. You can also pass `raise_exceptions=False` incase you want to show only a warning message instead.

## RAGAS feature extraction results analysis

Load features and labels

In [ ]:
feature_dfs = []
for i in range(N_TIMES):
    feature_path = OUTPUT_DIR / f"merged_ragas_features_run_{i+1}.csv"
    feature_df = pd.read_csv(feature_path)
    feature_dfs.append(feature_df)
print(f"\n=== Comparison of RAGAS features across {N_TIMES} runs ===")
for metric in metric_names:
    print(f"\n--- Metric: {metric} ---")
    metric_values = [df[metric] for df in feature_dfs]
    metric_means = [values.mean() for values in metric_values]
    metric_stds = [values.std() for values in metric_values]
    print(f"Means: {metric_means}")
    print(f"Stds: {metric_stds}")
    
    # Boxplot comparison
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=metric_values)
    plt.xticks(ticks=range(N_TIMES), labels=[f"Run {i+1}" for i in range(N_TIMES)])
    plt.title(f"Distribution of '{metric}' across runs")
    plt.ylabel(metric)
    plt.show()

(9806, 6)
